# Hafta 7 — Diziden Diziye Modeller ve Dikkat

Kodlayıcı–kod çözücü paradigması ve onun bilgi darboğazını kıran dikkat mekanizması. Bu, Transformer'a giden kavramsal köprüdür.

## Öğrenme Hedefleri

- Oyuncak bir çeviri görevi için LSTM tabanlı bir seq2seq modeli uygulamak.
- Sade seq2seq'in bilgi darboğazı başarısızlığını deneysel olarak belirlemek.
- Bahdanau (toplamsal) ve Luong (çarpımsal) dikkati türetmek ve uygulamak.
- Dikkat ısı haritalarını görselleştirmek ve dilbilimsel olarak yorumlamak.

## Zorunlu Okumalar

- Sutskever, I., Vinyals, O. ve Le, Q. V. (2014). *Sequence to Sequence Learning with Neural Networks*.
- Bahdanau, D., Cho, K. ve Bengio, Y. (2015). *Neural Machine Translation by Jointly Learning to Align and Translate*.
- Luong, M.-T., Pham, H. ve Manning, C. D. (2015). *Effective Approaches to Attention-based Neural Machine Translation*.

In [ ]:
import sys, math
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

np.random.seed(0)
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

## 1. Oyuncak bir çeviri görevi

Çevirinin yapısını veri yükü olmadan yakalayan küçük, sentetik bir görev tanımlıyoruz: İngilizce rakam adlarıyla yazılmış bir dizi → tersine çevrilmiş dizi. Örneğin: `one two three` → `three two one`.

Diziler uzadıkça bu, sade seq2seq için zorlaşır; çünkü kodlayıcı girdinin tamamını sabit boyutlu tek bir vektöre sıkıştırmak zorundadır.

In [ ]:
DIGITS = ['zero','one','two','three','four','five','six','seven','eight','nine']
PAD, BOS, EOS = '<pad>', '<s>', '</s>'
SPECIALS = [PAD, BOS, EOS]
VOCAB = SPECIALS + DIGITS
tok2id = {t: i for i, t in enumerate(VOCAB)}
id2tok = {i: t for t, i in tok2id.items()}
V = len(VOCAB)

def make_example(length):
    src = [np.random.randint(0, 10) for _ in range(length)]
    tgt = src[::-1]
    src_ids = [tok2id[DIGITS[d]] for d in src]
    tgt_ids = [tok2id[BOS]] + [tok2id[DIGITS[d]] for d in tgt] + [tok2id[EOS]]
    return src_ids, tgt_ids

def make_batch(N, length):
    pairs = [make_example(length) for _ in range(N)]
    src = torch.tensor([p[0] for p in pairs])      # (N, uzunluk)
    tgt = torch.tensor([p[1] for p in pairs])      # (N, uzunluk+2)
    return src, tgt

src, tgt = make_batch(2, 5)
print(f"src: {[id2tok[i.item()] for i in src[0]]}")
print(f"tgt: {[id2tok[i.item()] for i in tgt[0]]}")

## 2. Sade seq2seq

Kodlayıcı kaynağı okur ve son bir gizli durum $\mathbf{s}$ üretir. Kod çözücü $\mathbf{s}$ ile başlatılır ve hedefi öz-bağlanımlı olarak üretir. Kod çözücünün kaynak hakkında öğreneceği her şey bu tek vektöre sığmak zorundadır — **darboğaz** budur.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, V, emb=32, hidden=64):
        super().__init__()
        self.emb = nn.Embedding(V, emb, padding_idx=tok2id[PAD])
        self.rnn = nn.LSTM(emb, hidden, batch_first=True)

    def forward(self, src):
        e = self.emb(src)
        out, (h, c) = self.rnn(e)
        return out, (h, c)

class DecoderNoAttn(nn.Module):
    def __init__(self, V, emb=32, hidden=64):
        super().__init__()
        self.emb = nn.Embedding(V, emb, padding_idx=tok2id[PAD])
        self.rnn = nn.LSTM(emb, hidden, batch_first=True)
        self.out = nn.Linear(hidden, V)

    def forward(self, tgt_in, state):
        e = self.emb(tgt_in)
        out, state = self.rnn(e, state)
        return self.out(out), state

class Seq2SeqNoAttn(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = Encoder(V); self.dec = DecoderNoAttn(V)

    def forward(self, src, tgt_in):
        _, state = self.enc(src)
        logits, _ = self.dec(tgt_in, state)
        return logits

## 3. Bahdanau (toplamsal) dikkat

Her kod çözücü adımı $t$'de, her kodlayıcı durumu $\mathbf{h}_i$'ye karşı hizalama skorları hesapla:

$$e_{t,i} = \mathbf{v}^\top \tanh(W_s \mathbf{s}_{t-1} + W_h \mathbf{h}_i),$$

$$\alpha_{t,i} = \frac{\exp(e_{t,i})}{\sum_j \exp(e_{t,j})}, \qquad \mathbf{c}_t = \sum_i \alpha_{t,i} \mathbf{h}_i.$$

Bağlam vektörü $\mathbf{c}_t$, bir sonraki gizli durumu üretmek için kod çözücü girdisiyle birleştirilir. Kod çözücü artık tek bir son kodlayıcı durumuyla sınırlı değildir — herhangi bir konuma dikkat edebilir.

In [ ]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.Ws = nn.Linear(hidden, hidden, bias=False)
        self.Wh = nn.Linear(hidden, hidden, bias=False)
        self.v  = nn.Linear(hidden, 1, bias=False)

    def forward(self, decoder_state, encoder_outputs, src_mask=None):
        # decoder_state: (B, H);  encoder_outputs: (B, S, H);  src_mask: (B, S) bool, PAD konumlarında True
        s = self.Ws(decoder_state).unsqueeze(1)           # (B, 1, H)
        h = self.Wh(encoder_outputs)                       # (B, S, H)
        e = self.v(torch.tanh(s + h)).squeeze(-1)         # (B, S)
        if src_mask is not None:
            e = e.masked_fill(src_mask, -1e9)
        alpha = F.softmax(e, dim=-1)                       # (B, S)
        ctx = torch.bmm(alpha.unsqueeze(1), encoder_outputs).squeeze(1)  # (B, H)
        return ctx, alpha

class DecoderAttn(nn.Module):
    def __init__(self, V, emb=32, hidden=64):
        super().__init__()
        self.emb = nn.Embedding(V, emb, padding_idx=tok2id[PAD])
        self.attn = BahdanauAttention(hidden)
        self.rnn = nn.LSTMCell(emb + hidden, hidden)
        self.out = nn.Linear(hidden * 2, V)
        self.hidden = hidden

    def forward(self, tgt_in, enc_out, state, src_mask=None):
        # Dikkati okunabilir tutmak için adım adım kod çözme.
        h, c = state
        h, c = h.squeeze(0), c.squeeze(0)   # (B, H)
        B, T = tgt_in.shape
        logits_all, alphas_all = [], []
        for t in range(T):
            e = self.emb(tgt_in[:, t])
            ctx, alpha = self.attn(h, enc_out, src_mask)
            h, c = self.rnn(torch.cat([e, ctx], dim=-1), (h, c))
            logits = self.out(torch.cat([h, ctx], dim=-1))
            logits_all.append(logits); alphas_all.append(alpha)
        return torch.stack(logits_all, dim=1), torch.stack(alphas_all, dim=1)

class Seq2SeqAttn(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = Encoder(V); self.dec = DecoderAttn(V)

    def forward(self, src, tgt_in):
        enc_out, (h, c) = self.enc(src)
        src_mask = (src == tok2id[PAD])
        logits, alphas = self.dec(tgt_in, enc_out, (h, c), src_mask)
        return logits, alphas

## 4. Eğitim: uzun dizilerde dikkatli ve dikkatsiz model

Her iki modeli de tersine çevirme görevinde 4 ile 16 arasındaki uzunluklarda eğitiyor, ardından 4, 8, 12 ve 16 uzunluklarında değerlendiriyoruz. Tüm bu uzunluklar eğitim dağılımının içindedir; yani burada ölçtüğümüz şey uzunluğa genelleme değil, **saf darboğaz etkisidir**: dizi uzadıkça dikkatsiz modelin tek vektörlük özeti yetersiz kalır, dikkatli model ise her adımda kaynağın ilgili konumuna doğrudan bakabilir.

(Eğitimde hiç görülmemiş uzunluklara *genelleme* ayrı ve daha zor bir sorudur — bkz. 1. alıştırma.)

> Bu hücrenin CPU'da çalışması yaklaşık 30–40 saniye sürer.

In [ ]:
def train(model, lengths_train=(4,), n_steps=600, batch=64):
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    for step in range(n_steps):
        L = lengths_train[step % len(lengths_train)]
        src, tgt = make_batch(batch, L)
        tgt_in, tgt_out = tgt[:, :-1], tgt[:, 1:]
        out = model(src, tgt_in)
        logits = out[0] if isinstance(out, tuple) else out
        loss = F.cross_entropy(logits.reshape(-1, V), tgt_out.reshape(-1),
                               ignore_index=tok2id[PAD])
        opt.zero_grad(); loss.backward(); opt.step()
    return model

def evaluate(model, length, n=200):
    src, tgt = make_batch(n, length)
    tgt_in, tgt_out = tgt[:, :-1], tgt[:, 1:]
    with torch.no_grad():
        out = model(src, tgt_in)
        logits = out[0] if isinstance(out, tuple) else out
        preds = logits.argmax(-1)
    # Dizi düzeyinde doğruluk: PAD olmayan tüm konumlar doğru olmalı.
    mask = (tgt_out != tok2id[PAD])
    seq_correct = ((preds == tgt_out) | ~mask).all(dim=1).float().mean().item()
    return seq_correct

TRAIN_LENGTHS = tuple(range(4, 17))   # 4..16
m_noattn = train(Seq2SeqNoAttn(), lengths_train=TRAIN_LENGTHS, n_steps=1500)
m_attn   = train(Seq2SeqAttn(),   lengths_train=TRAIN_LENGTHS, n_steps=1500)

print(f"{'uzunluk':>8} {'dikkatsiz':>10} {'dikkatli':>10}")
for L in [4, 8, 12, 16]:
    a1 = evaluate(m_noattn, L); a2 = evaluate(m_attn, L)
    print(f"{L:>8d} {a1:>10.3f} {a2:>10.3f}")
print("\nDizi uzadıkça dikkatsiz modelin doğruluğu çöker, dikkatli modelinki yüksek kalır — darboğaz hikâyesi budur.")

## 5. Dikkati görselleştirmek

Başarılı bir çeviride dikkat, $i$. çıktı rakamını $(L − i)$. girdi rakamıyla hizalamalıdır (çünkü görev tersine çevirmedir). Isı haritası belirgin bir ters köşegen göstermelidir.

In [ ]:
src, tgt = make_batch(1, 6)
tgt_in = tgt[:, :-1]
with torch.no_grad():
    _, alphas = m_attn(src, tgt_in)
alpha = alphas[0].numpy()  # (T_hedef, T_kaynak)

src_tokens = [id2tok[i.item()] for i in src[0]]
tgt_tokens = [id2tok[i.item()] for i in tgt_in[0]]

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(alpha, cmap='viridis', aspect='auto')
ax.set_xticks(range(len(src_tokens))); ax.set_xticklabels(src_tokens)
ax.set_yticks(range(len(tgt_tokens))); ax.set_yticklabels(tgt_tokens)
ax.set_xlabel('kaynak'); ax.set_ylabel('hedef (kod çözücü adımı)')
ax.set_title('Bahdanau dikkati — tersine çevirmede ters köşegen göstermeli')
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

## 6. Luong dikkati — çarpımsal varyantlar

Luong vd. (2015) daha basit skor fonksiyonları önerdi:

- **Nokta (dot):** $e_{t,i} = \mathbf{s}_t^\top \mathbf{h}_i$
- **Genel (general):** $e_{t,i} = \mathbf{s}_t^\top W \mathbf{h}_i$
- **Birleştirme (concat):** Bahdanau'ya benzer

Nokta varyantı ek parametre gerektirmez ve daha hızlıdır — 8. haftadaki Transformer'ın **ölçekli nokta çarpımı dikkatinin** doğrudan habercisidir.

In [ ]:
class LuongDotAttention(nn.Module):
    def forward(self, decoder_state, encoder_outputs, src_mask=None):
        # decoder_state: (B, H);  encoder_outputs: (B, S, H)
        e = torch.bmm(encoder_outputs, decoder_state.unsqueeze(-1)).squeeze(-1)  # (B, S)
        if src_mask is not None:
            e = e.masked_fill(src_mask, -1e9)
        alpha = F.softmax(e, dim=-1)
        ctx = torch.bmm(alpha.unsqueeze(1), encoder_outputs).squeeze(1)
        return ctx, alpha

# Akıl sağlığı denetimi: Bahdanau dikkatiyle aynı boyut kuralları.
attn = LuongDotAttention()
B, S, H = 2, 6, 64
fake_state = torch.randn(B, H); fake_enc = torch.randn(B, S, H)
ctx, a = attn(fake_state, fake_enc)
print(f"bağlam boyutu: {tuple(ctx.shape)}, dikkat boyutu: {tuple(a.shape)}, toplam α = {a.sum(-1).tolist()}")

## 7. Alıştırmalar

1. **Uzunluğa genelleme.** Dikkatli ve dikkatsiz seq2seq'i $\{3, 4, 5\}$ uzunluklarında eğitin. 20'ye kadar uzunluklarda değerlendirin. Her ikisi için doğruluğu uzunluğa göre çizdirin. Eğitimde görülmemiş uzunluklarda iki model de neden zorlanıyor? (İpucu: dizi sonu tokenı `</s>`'nin konumu ve konumsal bilginin nasıl kodlandığı.)
2. **Monoton olmayan hizalama.** Monoton olmayan hizalamaya sahip küçük bir sentetik görev kurun (ör. tersine çevirme olmayan bir permütasyon). Dikkat ısı haritasını görselleştirin ve gerçek permütasyonu kurtarıp kurtarmadığını kontrol edin.
3. **Kapsama (coverage) dikkati.** Kapsama dikkatini uygulayın (Tu vd., 2016): dikkat skorlayıcısına girdi olarak bir kapsama vektörü $\mathbf{c}_t = \sum_{t' < t} \boldsymbol{\alpha}_{t'}$ ekleyin. Fazla ve eksik çeviriyi azalttığını gösterin.
4. **Bahdanau ve Luong'u karşılaştırın.** Her ikisini de eşleştirilmiş parametre sayılarıyla aynı veride eğitin. Nihai BLEU'yu ve eğitim süresini raporlayın.

---

## Gelecek Hafta

Hafta 8 — Uçtan uca Transformer. Yinelemeyi saf dikkatle değiştiriyoruz.